# Legacy trajectory gözlemcisi — devam hücreleri (2026-10-11)

`scripts/gsd_guidance_ablations_colab.ipynb` içindeki ilk **6 code hazırlık hücresinden sonra**, aynı hazır GPU runtime'da kullanın. Eski notebook'un sonraki deney hücrelerini çalıştırmayın. Mevcut notebook değiştirilmedi.

Baseline: Legacy B70/M800, spectral1.17/SCD1, beta2, gamma=eta=.01, lambda=.95, normal10/Background30. Point-MAE ağırlıkları/BN/token akışı sabit. Yerelde GPU koşusu yapılmadı. GPU smoke/pilot çağrılarını kullanıcı başlatır. **Run all kullanmayın.** Smoke ve pilot ayrı duraklardır.

Güncel plan: önce tüm15 corruption için280 örnek üzerinde **seed0**. Fayda görülürse kuralı sabitleyip seed1/2 ile tekrarlanabilirliğini ayrıca değerlendireceğiz; sayısal bir terfi eşiği henüz yok.


## 7 — GitHub branch güncellemesi ve kaynak doğrulaması

İlk6 hücre `dev` branch'ini bırakabiliyor. Kodlar `origin/gsd-smooth-spectrum` üzerinde; kaynak ZIP'i yüklemeyin. Aşağıdaki geçiş/pull yerel değişiklikle çatışırsa durur; reset/clean uygulanmaz. Daha önce kaynak ZIP'i açtıysanız çatışan dosyaları silmeden önce ayrı yedekleyin ve mevcut farkları inceleyin. Derlemenin ürettiği build/lib farkları engellenmez. Doğrulama yalnızca tanımlı inference kaynaklarını ve provenance dosyasını mevcut Git commit'iyle karşılaştırır; session commit/SHA kaydını saklar.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
git status --short
git fetch origin gsd-smooth-spectrum
git switch gsd-smooth-spectrum
git pull --ff-only origin gsd-smooth-spectrum
git rev-parse HEAD


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py verify


## 8 — Sabit indeks manifesti ve CPU preview (GPU başlatmaz)

Etikete bakmadan seed20261011 ile 280 indeks seçilir; Bu ilk aşamada yalnızca deney seed0 kullanılır; seed1/2 fayda incelemesinden sonraya bırakılır. İndeks seçme seed20261011 değişmez. Smoke bu listenin ilk140 indeksidir (iki B70). Pilot her corruption için dört B70: 15×280×1 = **4.200 terminal sınıflandırma**, her örnekte ek üç clean probe. Development kapsamıdır; all2468 benchmark değildir. Checkpoint ve indeks manifestleri koşudan önce yazılır.

Güncellenmiş plan için yeni session oluşturun; önceki sonuçları koruyun. Önceden tamamlanmış smoke varsa yeniden başlatmadan önce mevcut kanıtın yeniden kullanımını inceleyelim. Aktif session sırasında branch/commit veya inference kaynaklarını değiştirmeyin.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="/content/3DD-TTA/result/modelnet40_c/legacy_trajectory/$(date -u +%Y%m%dT%H%M%SZ)_trajectory"
printf '%s' "$SESSION" > /content/legacy_trajectory_session.txt
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py init --session "$SESSION"
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py preview --session "$SESSION"


## 9 — GPU smoke; ardından DUR

Gaussian ve Background, seed0, 140 örnek/corruption, üç mod: off / audit (yalnızca state+terminal) / observe. Toplam840 örnek işleme. Bu yeni gözlemcinin entegrasyon testidir; eski benchmark tekrarı veya accuracy kanıtı değildir.

Her arm kendi içinde tamamlanmış indeksleri, finite40 logits'i, model state/buffer değişmezliğini doğrular. Audit kolu aynı VAE encode'u aynı pre-encode RNG durumuyla tekrarlar; latent eşitliği ve buffer durumu kaydedilir. Observe kolunda her probe'un RNG/buffer öncesi-sonrası eşitliği kontrol edilir. Ayrı GPU süreçleri farklı latent/logit üretebilir; bu fark raporlanır, smoke'u tek başına başarısız yapmaz.

Geometry: ilk örnek/probe/batch, LION decoder koordinatlarında; pilotta kapalı. Her mühürlenen arm ZIP'i (başarısız olsa da) Drive'a SHA doğrulanarak yedeklenir. OOM'da batch/horizon değiştirmeyin. Hata sonrasında da hücre10 ile eldeki tam kanıtı indirin. Başarılı smoke sonrasında DURUN ve tam ZIP'i inceleme için paylaşın.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="$(cat /content/legacy_trajectory_session.txt)"
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py smoke --session "$SESSION" --drive-dir /content/drive/MyDrive/thesis/results/legacy_trajectory



## 10 — Sonuç ZIP'i, Drive yedeği ve indirme (smoke/pilot/hata sonrası)

Her çalıştırma yeni handoff ZIP oluşturur; önceki ZIP/dizinler korunur. Başarısız/kısmi çıktılar dahildir. Yalnızca yüzdeleri değil, ham logits/logları içeren tam ZIP'i paylaşın. Drive: `MyDrive/thesis/results/legacy_trajectory/`. İndirme hücresini tarayıcı indirmesine izin vererek çalıştırın. Tekrar aynı çıktı dizinine koşmayın; kısmi pilotta tamamlanan seed'leri yeniden çalıştırmadan önce eksik kapsamı inceleyelim.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="$(cat /content/legacy_trajectory_session.txt)"
HANDOFF="${SESSION}_handoff_$(date -u +%Y%m%dT%H%M%SZ).zip"
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py pack --session "$SESSION" --output "$HANDOFF"
printf '%s' "$HANDOFF" > /content/legacy_trajectory_handoff.txt
conda run --no-capture-output -n 3dd_tta_env python - <<'PY'
from pathlib import Path
from scripts.legacy_trajectory_colab import backup_zip
path = Path('/content/legacy_trajectory_handoff.txt').read_text()
backup_zip(Path(path), Path('/content/drive/MyDrive/thesis/results/legacy_trajectory'))
print('Verified Drive backup:', Path(path).name)
PY


In [ ]:
from pathlib import Path
from google.colab import files
files.download(Path('/content/legacy_trajectory_handoff.txt').read_text())


## 11 — Pilot: smoke incelemesinden sonra ayrı çalıştırın; ardından DUR

Runner teknik smoke kapısını doğrular; bilimsel incelemenin yerine geçmez. Smoke ile aynı Git commit ve doğrulanmış kaynaklar kullanılmalıdır. Pilot: tüm15, sabit280, yalnızca seed0, orijinal10/30 horizon. Seed1/2 ayrı, koşullu bir sonraki aşamadır. Politikalar aynı trace üzerinde hesaplanır; ek all15 baseline yeniden koşulmaz. Tek sealed pilot_seed0.zip + CPU analysis oluşur. Sonra hücre10'u tekrar çalıştırıp indirin/paylaşın. Tam benchmark veya erken durdurma otomatik başlatılmaz.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="$(cat /content/legacy_trajectory_session.txt)"
conda run --no-capture-output -n 3dd_tta_env python scripts/legacy_trajectory_colab.py pilot --session "$SESSION" --drive-dir /content/drive/MyDrive/thesis/results/legacy_trajectory


## 12 — Bağımsız CPU analyzer (opsiyonel; yeni GPU koşusu gerektirmez)

Runner zaten `pilot_analysis` üretir. Yeniden analiz gerekiyorsa yeni çıktı dizini kullanın. Terminal / minimum entropy / eşit sınıf olasılığı ortalaması, sabit probe1/2/3, corrected/broken, oracle tanısı ve corruption içinde original-index bootstrap kayıtları üretilir. Tek seed ile seed SD null kalır; seed kararlılığı veya genel istatistiksel anlamlılık iddiası kurulmaz. Tie-break: terminal, sonra en erken gözlem. Düşük entropy doğruluk veya temporal kararlılık garantisi değildir; oracle etiket kullanır ve inference politikası değildir. Timesteps bağımsız örnek değildir.


In [ ]:
%%bash
set -euo pipefail
cd /content/3DD-TTA
SESSION="$(cat /content/legacy_trajectory_session.txt)"
conda run --no-capture-output -n 3dd_tta_env python scripts/analyze_legacy_trajectory.py "$SESSION/pilot_seed0.zip" --output "${SESSION}_analysis_$(date -u +%Y%m%dT%H%M%SZ)"


Bu aşamada erken çıkış uygulanmaz; hızlanma iddiası yoktur. Senkronize bileşen süreleri ve toplamlar gözlemci/hash/I/O maliyetini içerir; saf inference benchmark'ı değildir. B1 canlı erken çıkış, yalnızca gözlenen prefix'e dayalı kural, aynı B1 full-horizon'a karşı ek probe dahil net latency ve bilinmeyen corruption için horizon sonraki açık konulardır. B70 throughput ayrı raporlanır. PG/BN/token değişikliği, M1800 ve başarısız Background100/1000 tekrarları kapsam dışıdır.
